In [6]:
import pandas as pd
import numpy as np

In [ ]:
produtos = pd.read_csv('../datasets/produtos_v2.csv')
interacoes = pd.read_csv('../datasets/interacoes_v3.csv')

produtos_new = pd.read_parquet('../datasets/produtos_v2.csv')
interacoes_new = pd.read_parquet('../datasets/interacoes_v3.csv')

In [19]:
produtos.head(5)

,id_produto,nome,categoria,preco_medio,perecibilidade_dias
0,1,Arroz 5kg,Grãos,25.0,365
1,2,Feijão 1kg,Grãos,8.5,180
2,3,Macarrão 500g,Massas,4.0,365
3,4,Leite Integral 1L,Laticínios,5.5,120
4,5,Óleo de Soja 900ml,Óleos,7.0,365


In [8]:
produtos.shape

(20, 5)

In [9]:
interacoes.head(5)

,id_usuario_produto,id_usuario,id_produto,id_estabelecimento,quantidade,distancia_km,timestamp
0,1,319,18,6,0.62,2.19,2026-02-20 17:00:00
1,2,289,1,33,0.88,4.78,2026-02-24 10:00:00
2,3,444,15,6,1.32,3.79,2026-02-03 14:00:00
3,4,168,7,17,2.55,3.19,2026-02-13 20:00:00
4,5,451,13,31,1.75,1.20,2026-02-23 12:00:00


In [10]:
interacoes.shape

(5000, 7)

In [11]:

ids_produtos = np.sort(produtos['id_produto'].unique())

relacionamentos_usuario = {}

In [12]:

for id_usuario, df_user in interacoes.groupby('id_usuario'):
    df_user = df_user.copy()

    media_dist = df_user['distancia_km'].mean()
    desvio_dist = df_user['distancia_km'].std()
    distancia_maxima = media_dist + 2 * desvio_dist

    df_user['distancia_tratada'] = df_user['distancia_km'].clip(upper=distancia_maxima)

    min_dist = df_user['distancia_tratada'].min()
    max_dist = df_user['distancia_tratada'].max()

    if pd.isna(max_dist) or pd.isna(min_dist):
        df_user['distancia_normalizada'] = 0.0
    elif max_dist == min_dist:
        df_user['distancia_normalizada'] = 0.0
    else:
        df_user['distancia_normalizada'] = (
            (df_user['distancia_tratada'] - min_dist) / (max_dist - min_dist)
        )

    df_user['score'] = df_user['distancia_normalizada'] + 1

    df_user = (
        df_user.groupby('id_produto', as_index=False)['score']
        .sum()
    )

    df_user = (
        pd.DataFrame({'id_produto': ids_produtos})
        .merge(df_user, on='id_produto', how='left')
    )

    df_user['score'] = df_user['score'].fillna(0)

    df_user = df_user.sort_values('id_produto').reset_index(drop=True)

    relacionamentos_usuario[id_usuario] = df_user

matriz_produto_usuario = pd.concat(
    [
        df_user.set_index('id_produto')['score'].rename(id_usuario)
        for id_usuario, df_user in relacionamentos_usuario.items()
    ],
    axis=1
).sort_index()




In [13]:
matriz_produto_usuario = matriz_produto_usuario.reset_index()

In [14]:
matriz_produto_usuario

,id_produto,1,2,3,4,5,6,7,8,9,...,491,492,493,494,495,496,497,498,499,500
0,1,0.000000,0.000000,0.000000,0.000000,0.0000,1.343249,1.564607,1.103960,0.000000,...,0.000000,0.000000,1.649123,0.000000,1.670300,1.000000,0.000000,0.000000,1.572093,1.782895
1,2,0.000000,1.024570,0.000000,2.000000,0.0000,0.000000,1.671348,1.678218,0.000000,...,5.016173,0.000000,1.087719,1.477273,0.000000,0.000000,5.234637,0.000000,0.000000,0.000000
2,3,0.000000,0.000000,0.000000,0.000000,2.0325,1.228833,1.000000,3.596535,0.000000,...,0.000000,0.000000,1.288221,0.000000,0.000000,0.000000,0.000000,3.000000,3.234884,0.000000
3,4,0.000000,1.393120,1.789116,0.000000,0.0000,6.121281,0.000000,1.970297,0.000000,...,1.000000,2.967816,0.000000,0.000000,0.000000,1.707165,0.000000,0.000000,0.000000,1.000000
4,5,0.000000,1.206388,0.000000,1.663333,0.0000,0.000000,1.404494,1.034653,3.129518,...,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.310056,0.000000,0.000000,0.000000
5,6,0.000000,1.272727,0.000000,0.000000,0.0000,1.466819,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,1.343324,3.333333,2.000000,1.772959,4.397674,0.000000
6,7,1.739857,2.909091,0.000000,0.000000,0.0000,2.000000,1.772472,0.000000,1.746988,...,0.000000,0.000000,2.000000,0.000000,2.463215,0.000000,1.298883,0.000000,1.646512,0.000000
7,8,0.000000,0.000000,0.000000,0.000000,1.8725,0.000000,0.000000,0.000000,0.000000,...,1.070081,1.342529,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000,1.300000,0.000000
8,9,0.000000,1.167076,1.000000,0.000000,1.6750,3.327231,2.000000,0.000000,1.852410,...,0.000000,1.144828,0.000000,0.000000,1.697548,0.000000,0.000000,0.000000,1.976744,0.000000
9,10,0.000000,0.000000,0.000000,0.000000,5.2775,0.000000,2.907303,0.000000,1.831325,...,0.000000,0.000000,0.000000,2.700758,1.324251,0.000000,1.687151,0.000000,0.000000,0.000000


In [15]:
matriz_produto_usuario.to_csv('../datasets/matriz_produto_usuario.csv', index=False)

In [16]:
count_zeros = pd.DataFrame()
count_zeros['count_zeros'] = (matriz_produto_usuario == 0).sum(axis=1)/len(matriz_produto_usuario.columns)
count_zeros

,count_zeros
0,0.608782
1,0.580838
2,0.604790
3,0.590818
4,0.602794
5,0.586826
6,0.610778
7,0.642715
8,0.624750
9,0.634731


In [17]:
X = matriz_produto_usuario.drop(columns='id_produto').to_numpy(dtype=float)
n_produtos, n_usuarios = X.shape

print(f"Dimensões: {n_produtos} produtos × {n_usuarios} usuários")
print(f"Total de células: {n_produtos * n_usuarios}")
print()

zeros = (X == 0).sum()
nao_zeros = (X != 0).sum()
esparsidade = zeros / (n_produtos * n_usuarios)

print(f"Células com valor zero: {zeros} ({esparsidade:.1%})")
print(f"Células com valor > 0: {nao_zeros} ({1 - esparsidade:.1%})")
print()

print("Estatísticas dos scores (somente valores > 0):")
scores_nao_zero = X[X != 0]
print(pd.Series(scores_nao_zero).describe(percentiles=[0.25, 0.5, 0.75]).round(4))
print()

compradores_por_produto = (X != 0).sum(axis=1)
print("Usuários compradores por produto:")
print(pd.Series(compradores_por_produto).describe(percentiles=[0.25, 0.5, 0.75]).round(2))
print()

produtos_por_usuario = (X != 0).sum(axis=0)
print("Produtos distintos por usuário:")
print(pd.Series(produtos_por_usuario).describe(percentiles=[0.25, 0.5, 0.75]).round(2))


Dimensões: 20 produtos × 500 usuários
Total de células: 10000

Células com valor zero: 6059 (60.6%)
Células com valor > 0: 3941 (39.4%)

Estatísticas dos scores (somente valores > 0):
count    3941.0000
mean        1.9041
std         0.8814
min         1.0000
25%         1.2907
50%         1.6891
75%         2.0000
max         7.7899
dtype: float64

Usuários compradores por produto:
count     20.00
mean     197.05
std        9.54
min      178.00
25%      193.50
50%      197.00
75%      203.25
max      215.00
dtype: float64

Produtos distintos por usuário:
count    500.00
mean       7.88
std        2.25
min        1.00
25%        6.00
50%        8.00
75%        9.00
max       14.00
dtype: float64
